# Pattern — Plan-and-Execute

**Plan-and-execute** splits work into two phases: a **planner** produces an explicit multi-step plan, then an **executor** runs each step — often with tools — without re-deciding the whole strategy every turn.

```
  User goal / release task
         │
         ▼
  ┌──────────────┐
  │   PLANNER    │  "Step 1: lookup order … Step 2: check policy …"
  └──────┬───────┘
         │ ExecutionPlan
         ▼
  ┌──────────────┐
  │  EXECUTOR    │  run step → observe → next step
  └──────┬───────┘
         │ (optional replan if step fails)
         ▼
     Final result
```

Contrast:

| Pattern | Planning |
|---------|----------|
| **Prompt chain** | Fixed steps, no plan object |
| **Plan-and-execute** | Explicit plan, then execute |
| **ReAct** | Re-plan every loop iteration |

This notebook implements **ReleaseFlow** rollout planning and **ShopCo** multi-step ticket resolution in plain Python — with replanning, human gates, and execution traces.

In [ ]:
"""
Plan-and-Execute pattern lab — ShopCo + ReleaseFlow.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
MAX_REPLANS = 2


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days with receipt. [pol-returns-01]",
    "shipping": "Free shipping on orders over $50. [pol-shipping-02]",
    "refunds": "Refunds over $1000 require manager approval. [pol-refunds-03]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "total_usd": 89.0},
    "ORD-1002": {"order_id": "ORD-1002", "status": "delivered", "total_usd": 1299.0},
}

RELEASE_CHECKS = {
    "unit_tests": {"status": "pass", "detail": "412 tests passed"},
    "lint": {"status": "pass", "detail": "no issues"},
    "security_scan": {"status": "warn", "detail": "1 medium CVE"},
}

print("Plan-and-execute lab ready.")

---

## 1. When Plan-and-Execute Fits

| Use plan-and-execute | Prefer something simpler |
|----------------------|--------------------------|
| Multi-step SOP with named actions | Fixed 2-step chain → prompt chain |
| Long horizon, plan reviewed before run | One tool call → one-shot tool |
| Human approves plan upfront | Disjoint intents → routing |
| Replan only on failure, not every step | Unknown tool order → ReAct |

The plan is a **first-class artifact** you can log, audit, and show to operators.

---

## 2. Core Types — Plan Step and Execution State

In [ ]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"
    FAILED = "failed"
    SKIPPED = "skipped"
    AWAITING_HUMAN = "awaiting_human"


@dataclass
class PlanStep:
    step_id: str
    action: str
    description: str
    args: dict[str, Any] = field(default_factory=dict)
    requires_human: bool = False
    status: StepStatus = StepStatus.PENDING
    result: Any = None
    error: str | None = None


@dataclass
class ExecutionPlan:
    plan_id: str
    goal: str
    steps: list[PlanStep]
    created_at: str = field(default_factory=utc_now)
    replan_count: int = 0

    def pending_steps(self) -> list[PlanStep]:
        return [s for s in self.steps if s.status == StepStatus.PENDING]

    def is_complete(self) -> bool:
        return all(s.status in (StepStatus.SUCCESS, StepStatus.SKIPPED) for s in self.steps)


demo = ExecutionPlan(
    plan_id="plan-demo", goal="Resolve return question",
    steps=[PlanStep("s1", "lookup_order", "Fetch order record", {"order_id": "ORD-1001"})],
)
print(demo.goal, "→", demo.steps[0].action)

---

## 3. Tool Registry — Actions the Executor Can Run

In [ ]:
def tool_lookup_order(order_id: str) -> dict[str, Any]:
    return ORDERS_DB.get(order_id, {"order_id": order_id, "status": "not_found", "total_usd": 0})


def tool_search_policy(topic: str) -> str:
    return POLICY_SNIPPETS.get(topic, "Policy not found.")


def tool_run_check(check_name: str) -> dict[str, str]:
    return RELEASE_CHECKS.get(check_name, {"status": "fail", "detail": "unknown check"})


def tool_deploy(target: str, version: str) -> str:
    return f"deployed {version} to {target}"


def tool_draft_reply(facts: dict[str, Any]) -> str:
    policy = facts.get("policy", "")
    order = facts.get("order", {})
    return (
        f"Order {order.get('order_id', 'N/A')} is {order.get('status', 'unknown')}. "
        f"{policy}"
    )


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "lookup_order": tool_lookup_order,
    "search_policy": tool_search_policy,
    "run_check": tool_run_check,
    "deploy": tool_deploy,
    "draft_reply": tool_draft_reply,
}

print("Tools:", list(TOOL_REGISTRY.keys()))

---

## 4. Planner — ShopCo Return Ticket

In [ ]:
def plan_shopco_return(message: str) -> ExecutionPlan:
    oid_m = re.search(r"ORD-[0-9]{4}", message.upper())
    order_id = oid_m.group(0) if oid_m else ""
    steps = [
        PlanStep("s1", "lookup_order", "Load order facts", {"order_id": order_id}),
        PlanStep("s2", "search_policy", "Fetch return policy", {"topic": "returns"}),
    ]
    if order_id:
        order = ORDERS_DB.get(order_id, {})
        if order.get("total_usd", 0) > 1000:
            steps.append(PlanStep(
                "s3", "search_policy", "Check refund escalation policy",
                {"topic": "refunds"}, requires_human=True,
            ))
    steps.append(PlanStep("s4", "draft_reply", "Compose customer reply", {}))
    return ExecutionPlan(
        plan_id=f"plan-{uuid.uuid4().hex[:8]}",
        goal=f"Resolve: {message[:60]}",
        steps=steps,
    )


shopco_plan = plan_shopco_return("Can I return ORD-1002? It was expensive.")
print("Steps:", [(s.step_id, s.action, s.requires_human) for s in shopco_plan.steps])

---

## 5. Planner — ReleaseFlow Rollout

In [ ]:
def plan_release_rollout(version: str, env: str = "production") -> ExecutionPlan:
    steps = [
        PlanStep("r1", "run_check", "Unit tests", {"check_name": "unit_tests"}),
        PlanStep("r2", "run_check", "Lint", {"check_name": "lint"}),
        PlanStep("r3", "run_check", "Security scan", {"check_name": "security_scan"}),
        PlanStep("r4", "deploy", "Deploy to staging", {"target": "staging", "version": version}),
        PlanStep("r5", "deploy", f"Deploy to {env}", {"target": env, "version": version}, requires_human=(env == "production")),
    ]
    return ExecutionPlan(
        plan_id=f"rel-{uuid.uuid4().hex[:8]}",
        goal=f"Roll out {version} to {env}",
        steps=steps,
    )


release_plan = plan_release_rollout("2.4.0")
print("Release steps:", [s.description for s in release_plan.steps])

---

## 6. Executor Engine

In [ ]:
@dataclass
class ExecutionContext:
    facts: dict[str, Any] = field(default_factory=dict)
    trace: list[dict[str, Any]] = field(default_factory=list)
    human_approved: bool = False


@dataclass
class ExecutionResult:
    plan: ExecutionPlan
    context: ExecutionContext
    success: bool
    final_output: Any = None
    blocked_on_human: bool = False


class PlanExecutor:
    def __init__(self, tools: dict[str, Callable[..., Any]]) -> None:
        self.tools = tools

    def _run_step(self, step: PlanStep, ctx: ExecutionContext) -> None:
        step.status = StepStatus.RUNNING
        if step.requires_human and not ctx.human_approved:
            step.status = StepStatus.AWAITING_HUMAN
            ctx.trace.append({"step": step.step_id, "status": "awaiting_human", "action": step.action})
            return
        fn = self.tools.get(step.action)
        if fn is None:
            step.status = StepStatus.FAILED
            step.error = f"unknown action: {step.action}"
            return
        try:
            if step.action == "lookup_order":
                out = fn(step.args["order_id"])
                ctx.facts["order"] = out
            elif step.action == "search_policy":
                out = fn(step.args["topic"])
                ctx.facts.setdefault("policies", []).append(out)
                ctx.facts["policy"] = " ".join(ctx.facts.get("policies", []))
            elif step.action == "run_check":
                out = fn(step.args["check_name"])
                ctx.facts.setdefault("checks", []).append(out)
                if out.get("status") == "fail":
                    step.status = StepStatus.FAILED
                    step.error = out.get("detail", "check failed")
                    step.result = out
                    ctx.trace.append({"step": step.step_id, "status": "failed", "result": out})
                    return
            elif step.action == "deploy":
                out = fn(step.args["target"], step.args["version"])
            elif step.action == "draft_reply":
                out = fn(ctx.facts)
                ctx.facts["reply"] = out
            else:
                out = fn(**step.args)
            step.result = out
            step.status = StepStatus.SUCCESS
            ctx.trace.append({"step": step.step_id, "status": "success", "action": step.action, "result": out})
        except Exception as exc:
            step.status = StepStatus.FAILED
            step.error = str(exc)
            ctx.trace.append({"step": step.step_id, "status": "failed", "error": str(exc)})

    def execute(self, plan: ExecutionPlan, human_approved: bool = False) -> ExecutionResult:
        ctx = ExecutionContext(human_approved=human_approved)
        for step in plan.steps:
            if step.status not in (StepStatus.PENDING, StepStatus.AWAITING_HUMAN):
                continue
            self._run_step(step, ctx)
            if step.status == StepStatus.AWAITING_HUMAN:
                return ExecutionResult(plan, ctx, success=False, blocked_on_human=True)
            if step.status == StepStatus.FAILED:
                return ExecutionResult(plan, ctx, success=False)
        final = ctx.facts.get("reply") or ctx.facts.get("checks") or ctx.trace
        return ExecutionResult(plan, ctx, success=plan.is_complete(), final_output=final)


executor = PlanExecutor(TOOL_REGISTRY)

---

## 7. Demo — ShopCo Return (No Human Gate)

In [ ]:
plan_a = plan_shopco_return("Can I return ORD-1001?")
result_a = executor.execute(plan_a)
print("Success:", result_a.success)
print("Reply:", result_a.final_output)
print("Trace steps:", len(result_a.context.trace))

---

## 8. Demo — High-Value Order Blocks on Human Gate

In [ ]:
plan_b = plan_shopco_return("I want a refund for ORD-1002")
blocked = executor.execute(plan_b, human_approved=False)
print("Blocked on human:", blocked.blocked_on_human)
print("Awaiting step:", next(s for s in plan_b.steps if s.status == StepStatus.AWAITING_HUMAN).description)

approved = executor.execute(plan_b, human_approved=True)
print("\nAfter approval — success:", approved.success)
print("Reply:", approved.final_output)

---

## 9. Demo — ReleaseFlow Rollout Plan

In [ ]:
rel_plan = plan_release_rollout("2.4.0", env="production")
rel_blocked = executor.execute(rel_plan, human_approved=False)
print("Checks run:", len(rel_blocked.context.facts.get("checks", [])))
print("Blocked before prod deploy:", rel_blocked.blocked_on_human)

rel_done = executor.execute(rel_plan, human_approved=True)
print("Rollout complete:", rel_done.success)
print("Last trace:", rel_done.context.trace[-1])

---

## 10. Replanning on Failure

When a step fails, the planner may emit a **recovery plan** instead of aborting entirely.

In [ ]:
def plan_recovery(failed_step: PlanStep, version: str) -> list[PlanStep]:
    if failed_step.action == "run_check" and failed_step.args.get("check_name") == "security_scan":
        return [
            PlanStep("rx1", "run_check", "Re-run security after waiver", {"check_name": "security_scan"}),
            PlanStep("rx2", "deploy", "Deploy to staging", {"target": "staging", "version": version}),
        ]
    return []


class PlanAndExecuteRunner:
    def __init__(self, planner: Callable[..., ExecutionPlan], executor: PlanExecutor) -> None:
        self.planner = planner
        self.executor = executor

    def run(self, **planner_kwargs: Any) -> ExecutionResult:
        plan = self.planner(**planner_kwargs)
        result = self.executor.execute(plan, human_approved=planner_kwargs.get("human_approved", False))
        replans = 0
        while not result.success and replans < MAX_REPLANS:
            failed = next((s for s in plan.steps if s.status == StepStatus.FAILED), None)
            if not failed:
                break
            recovery = plan_recovery(failed, planner_kwargs.get("version", "2.4.0"))
            if not recovery:
                break
            plan.replan_count += 1
            replans += 1
            failed.status = StepStatus.SKIPPED
            plan.steps.extend(recovery)
            result = self.executor.execute(plan, human_approved=planner_kwargs.get("human_approved", False))
        return result


runner = PlanAndExecuteRunner(plan_release_rollout, executor)
print("Runner ready — replan on security_scan failure supported.")

---

## 11. Plan Validation Before Execute

Reject invalid plans **before** running expensive or risky steps.

In [ ]:
def validate_plan(plan: ExecutionPlan) -> list[str]:
    errors: list[str] = []
    if not plan.steps:
        errors.append("plan has no steps")
    actions = [s.action for s in plan.steps]
    for a in actions:
        if a not in TOOL_REGISTRY:
            errors.append(f"unknown action: {a}")
    if "deploy" in actions and "run_check" not in actions:
        errors.append("deploy without prior run_check")
    return errors


bad_plan = ExecutionPlan("bad", "oops", [PlanStep("x", "deploy", "skip checks", {"target": "prod", "version": "9"})])
print("Validation errors:", validate_plan(bad_plan))
print("Release plan valid:", validate_plan(release_plan) == [])

---

## 12. Plan-and-Execute vs ReAct — Same Ticket

| Aspect | Plan-and-execute | ReAct |
|--------|------------------|-------|
| Plan visibility | Full plan upfront | Emerges step by step |
| LLM calls | Planner + optional executor LLM | Every loop |
| Best for | SOPs, releases | Exploratory lookup |
| Human review | Approve plan before run | Harder to preview |

In [ ]:
def react_style(message: str) -> dict[str, Any]:
    trace = []
    facts: dict[str, Any] = {}
    if "return" in message.lower():
        trace.append("think: need policy")
        facts["policy"] = tool_search_policy("returns")
    m = re.search(r"ORD-[0-9]{4}", message.upper())
    if m:
        trace.append("think: need order")
        facts["order"] = tool_lookup_order(m.group(0))
    reply = tool_draft_reply(facts)
    return {"pattern": "react", "trace": trace, "reply": reply, "planner_calls": len(trace)}


msg = "Can I return ORD-1001?"
pe_plan = plan_shopco_return(msg)
pe_result = executor.execute(pe_plan)
react_result = react_style(msg)
print("Plan-and-execute steps:", len(pe_plan.steps), "| ReAct think steps:", react_result["planner_calls"])
print("PE reply:", pe_result.final_output)
print("ReAct reply:", react_result["reply"])

---

## 13. Execution Trace for Operators

In [ ]:
def render_plan_timeline(plan: ExecutionPlan) -> str:
    lines = [f"Plan {plan.plan_id}: {plan.goal}"]
    for s in plan.steps:
        icon = {"success": "✓", "failed": "✗", "awaiting_human": "⏸", "pending": "·"}.get(s.status.value, "?")
        lines.append(f"  {icon} [{s.step_id}] {s.action}: {s.description} ({s.status.value})")
    return "\n".join(lines)


print(render_plan_timeline(plan_b))

---

## 14. Optional Live LLM Planner

Set `USE_LIVE_LLM = True` to generate plan steps as JSON with `gpt-4o-mini`.

In [ ]:
def plan_with_live_llm(goal: str) -> ExecutionPlan | None:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    tools = list(TOOL_REGISTRY.keys())
    resp = llm.invoke([
        SystemMessage(content=(
            f"Create a JSON list of steps for goal. Each step: step_id, action (from {tools}), "
            "description, args dict. Return JSON only."
        )),
        HumanMessage(content=goal),
    ])
    try:
        raw = json.loads(str(resp.content))
        steps = [PlanStep(r["step_id"], r["action"], r["description"], r.get("args", {})) for r in raw]
        return ExecutionPlan(f"llm-{uuid.uuid4().hex[:6]}", goal, steps)
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


goal = "Roll out 2.4.0: run tests then deploy staging then production"
if USE_LIVE_LLM:
    llm_plan = plan_with_live_llm(goal)
    print(pretty(llm_plan.steps if llm_plan else "parse failed"))
else:
    print("Offline planner:", [s.action for s in plan_release_rollout("2.4.0").steps])

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Plan-and-execute for 2 fixed steps | Extra planner latency | Prompt chain |
| No plan validation | Risky deploy runs | `validate_plan()` gate |
| Stale plan after world change | Wrong actions | Replan on failure |
| Executor re-plans every step | You built ReAct | Separate planner role |
| No human gate on prod | Incident | `requires_human` steps |

---

## 16. Quiz

1. What two roles define plan-and-execute?
2. How does it differ from ReAct?
3. When does ShopCo add a `requires_human` step?
4. Why validate the plan before execution?
5. When should you replan instead of aborting?

---

## 17. Summary

**Plan-and-execute** makes the strategy explicit: **plan** → **execute** → optional **replan**. ReleaseFlow rollouts and ShopCo multi-step tickets fit this shape when steps are known upfront and operators may approve the plan before risky actions.

This notebook implemented `ExecutionPlan`, rule-based planners, `PlanExecutor` with human gates, recovery replanning, and plan validation — all in plain Python.

Use **prompt chaining** when the SOP never varies; use **ReAct** when tool order is unknown; use **plan-and-execute** when you need a reviewable plan and controlled execution.